# 04 · Инструменты

**Цель:** модель вызывает функцию, когда без неё не ответить, не вызывает, когда ответ и так известен, а результат вызова читает и использует в ответе. Инструменты игрушечные, но настоящие: `calc`, `word_count`, `reverse`, `roll_dice` считают по-честному. Поэтому правильный итог известен, и конечную реплику агента можно проверить, а не оценивать на глаз.

Два замера. **Первый шаг**, как в BFCL: recall — вызов совпал с эталоном по имени и аргументам на группах `tool` / `multi`; FPR — вызов на `direct`, где инструмент не нужен (irrelevance). **Полный цикл**, как в τ-bench: доведён ли ответ до верного результата, pass@1 при жадной генерации. Схема цикла — `books/00-basics.pdf`; отдельный читаемый вариант с префиллом — `tools/agent_loop.py`.

In [ ]:
from common import MODEL_ID, SYSTEM, DATA, RUNS, read_raw, tools_suite, fmt

import torch
from transformers import AutoModelForImageTextToText, AutoProcessor, Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model
from vlmkit import ChatCollator, load_jsonl, memory_report, preview, evaluate as ev
from vlmkit.compat import supported, first_accepted
from vlmkit.toolcalls import detect_style, parse_tool_calls, strip_thinking
from vlmkit.toytools import SCHEMA, run as run_tool

model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.bfloat16, device_map={"": 0},
    attn_implementation="sdpa", trust_remote_code=True,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True, max_pixels=1003520)
print(memory_report())

print("формат вызова у модели:", detect_style(processor.tokenizer.chat_template))

## Инструменты

Схема — то, что модель видит через ветку `tools` шаблона. Функции — то, что исполняет код. Модель порождает строку; имя из неё сопоставляется с функцией и запускается. Ошибка — тоже результат: она возвращается текстом.

In [ ]:
for t in SCHEMA:
    f = t["function"]
    print(f"{f['name']:11} {f['description']}  ({', '.join(f['parameters']['properties'])})")

print()
for name, args in [("calc", {"expression": "17*23"}), ("calc", {"expression": "2**10 - 1"}),
                   ("word_count", {"text": "гипотеза вырастает из проблемы"}),
                   ("reverse", {"text": "диплом"}), ("calc", {"expression": "import os"})]:
    print(f"{name}({args}) → {run_tool(name, args)!r}")

## Цикл

Модель порождает строку. Код разбирает из неё вызовы, исполняет, кладёт результат репликой `tool` и пересобирает промпт из всего списка сообщений. Память агента — этот список; сама модель между шагами ничего не помнит. Рассуждение выключено, иначе лимит токенов уйдёт в `<think>`.

In [ ]:
def solve(question, max_steps=4, verbose=False):
    """Вопрос → конечный ответ агента и число вызовов."""
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}]
    n_calls = 0
    for _ in range(max_steps):
        prompt = processor.apply_chat_template(messages, tools=SCHEMA, tokenize=False,
                                               add_generation_prompt=True, enable_thinking=False)
        inputs = processor(text=[prompt], return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
        reply = strip_thinking(processor.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True))
        messages.append({"role": "assistant", "content": reply})
        if verbose:
            print(f"  модель: {reply[:120]!r}")

        calls = parse_tool_calls(reply)
        if not calls:
            return reply, n_calls                  # ответила словами — цикл окончен
        for c in calls:
            result = run_tool(c["name"], c["arguments"])
            n_calls += 1
            messages.append({"role": "tool", "content": result})
            if verbose:
                print(f"  {c['name']}({c['arguments']}) → {result}")
    return "", n_calls                              # вышли по лимиту — модель не остановилась сама

## До

Базовая модель считает `17*23` в уме и часто права — на первом шаге инструмент не вызывает. На двухзначных умножениях и подсчёте слов она ошибается чаще, чем кажется; колонка «верный итог» это и показывает.

In [ ]:
suite = tools_suite()
predictions = ev.generate(model, processor, suite.samples, system=SYSTEM, tools=suite.tools)
before_metrics = suite.score(predictions)
print("первый шаг:", fmt(before_metrics))
print("по группам:", suite.rates(predictions), "— точные совпадения на tool/multi, вызовы на direct")

eval_rows = [r for r in read_raw("tools.jsonl")[1::2] if r["expected"]]

def solved_rate(rows):
    ok = 0
    for r in rows:
        answer, _ = solve(r["messages"][0]["content"][0]["text"])
        ok += all(v in answer for v in r["expected"])
    return ok / len(rows)

before_solved = solved_rate(eval_rows)
print(f"полный цикл: верный итог в {before_solved:.0%} задач из {len(eval_rows)}")

print("\nпример:")
solve("Сколько будет 17 * 23?", verbose=True)

## Что попадает в градиент

На траектории `multi` должно быть видно: обе реплики ассистента открыты, вопрос, описание инструментов и результаты вызовов закрыты, блок `<think>` закрыт.

In [ ]:
multi = next(s for s, r in zip(load_jsonl(DATA / "tools.jsonl"), read_raw("tools.jsonl")) if r["group"] == "multi")
print(preview(multi, processor, system=SYSTEM, tools=SCHEMA)[-1400:])

## Обучение

Коллатору передаётся тот же `SCHEMA`, что и при генерации: модель учится на том списке инструментов, с которым будет работать.

In [ ]:
lora = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=r"^(?!.*(visual|vision)).*(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)$",
    use_rslora=True,
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora)
model.print_trainable_parameters()

# Без этого при gradient checkpointing градиент не доходит до адаптеров.
# Ошибки не будет — обучение просто ничего не даст.
model.enable_input_require_grads()
model.config.use_cache = False

train = load_jsonl(DATA / "tools.jsonl")[::2]
collator = ChatCollator(processor, system=SYSTEM, tools=SCHEMA)

args = dict(
    output_dir=str(RUNS / "sft-tools"),
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=2,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    bf16=True,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch_fused",
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    report_to=[],
    seed=42,
)
args |= first_accepted(TrainingArguments, {"warmup_ratio": 0.05, "warmup_steps": 2})

trainer = Trainer(
    model=model,
    args=TrainingArguments(**supported(TrainingArguments, args)),
    train_dataset=train,
    data_collator=collator,
)
result = trainer.train()
print(f"loss {result.training_loss:.3f}, {result.metrics.get('train_runtime', 0)/60:.1f} мин")

## После

In [ ]:
model.eval()
after_metrics = ev.run(model, processor, suite)
after_solved = solved_rate(eval_rows)

print(f"первый шаг   до: {fmt(before_metrics)}")
print(f"          после: {fmt(after_metrics)}")
print(f"верный итог  до: {before_solved:.0%}   после: {after_solved:.0%}")

for q in ("Сколько будет 48 * 12?", "Переверни слово «антиплагиат».", "Что такое гипотеза?", "Брось кубик на 20 граней."):
    print(f"\n{q}")
    answer, n = solve(q, verbose=True)
    print(f"  итог ({n} вызовов): {answer[:120]!r}")

model.save_pretrained(str(RUNS / "sft-tools"))

## На что смотреть

**Вызовы есть, итог неверный** — модель не читает результат, а отвечает из головы. В обучении результат вызова замаскирован правильно, но проверьте, что конечная реплика в данных на него опирается.

**Вызовы на `direct`** — переобобщение, как и с матом: добавить примеров без инструмента.

**Формат вызова не разбирается** — сравните `detect_style` с тем, что порождает модель; данные собраны под XML.

**`roll_dice` без вызова** — модель «бросила» кубик сама. Ровно тот случай, ради которого результат инструмента маскируется в обучении: иначе она учится его сочинять.